# 05 — Adding `days_to_trend` to the corrected (dedup + grouped) feature set

`04_dedup_group_split.ipynb` fixed the video-identity leakage but flagged a remaining issue: after dedup, a video still contributes one row per distinct `trending_date`, all sharing identical *pre-publish* features (`video_category_id`, `tag_count`, `title_length`, `title_has_caps_word`, `publish_hour`, `publish_dayofweek`) but a target that climbs across those rows as the video accumulates views during its trending run (e.g. the Kendrick Lamar Super Bowl video: 16.3M views on day 1 of trending → 94.2M by day ~21). With no feature distinguishing an early-trending snapshot from a late one, that's irreducible label noise for the v2 feature set.

`build_features` in `src/features.py` already computes `days_to_trend` — `(trending_date - video_published_at)` in days — which is exactly that missing signal: how long the video had been public by the time this particular row's view-count snapshot was taken. Adding it directly tests the theory: if most of the remaining unexplained variance really is "which day of the trending run is this," `days_to_trend` should pick up a large share of it.

Same dedup (`video_id` + `trending_date`) and `GroupShuffleSplit` by `video_id` (`test_size=0.2, random_state=42`) as `04_dedup_group_split.ipynb` — only the feature set changes, so the comparison isolates the effect of `days_to_trend`.

In [1]:
import sys
sys.path.append('../src')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

from features import build_features

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

MODELS_DIR = Path('../models')

DATA_PATH = Path('../data/processed/train.parquet')
df = pd.read_parquet(DATA_PATH)
df = build_features(df)
print(f"Loaded and featurized {len(df):,} rows")

Loaded and featurized 9,876,866 rows


## Feature set — v2 pre-publish features + `days_to_trend`

In [2]:
FEATURE_COLUMNS_RAW = [
    'video_category_id',
    'tag_count',
    'title_length',
    'title_has_caps_word',
    'publish_hour',
    'publish_dayofweek',
    'days_to_trend',
    'video_trending_country',
]
FEATURE_COLUMNS_RAW_V3 = [c for c in FEATURE_COLUMNS_RAW if c != 'video_trending_country']
TARGET_COLUMN = 'video_view_count'
KEY_COLUMNS = ['video_id', 'trending_date']

model_df = df[FEATURE_COLUMNS_RAW + [TARGET_COLUMN] + KEY_COLUMNS].dropna(
    subset=FEATURE_COLUMNS_RAW + [TARGET_COLUMN]
).copy()
print(f"Rows before dedup: {len(model_df):,} (from {len(df):,})")

Rows before dedup: 9,876,866 (from 9,876,866)


## Dedup: one row per `(video_id, trending_date)`

Same procedure as `04_dedup_group_split.ipynb`. Note `days_to_trend` is itself a function of `(video_id, trending_date)`, so it does NOT collapse away in this dedup step the way country did — rows for the same video on different trending dates now have genuinely different `days_to_trend` values, not identical features.

In [3]:
model_df_sorted = model_df.sort_values(['video_id', 'trending_date', 'video_trending_country'])
deduped_df = model_df_sorted.drop_duplicates(subset=['video_id', 'trending_date'], keep='first').copy()

print(f"Rows before dedup: {len(model_df):,}")
print(f"Rows after dedup:  {len(deduped_df):,}")
print(f"Unique video_id in deduped data: {deduped_df['video_id'].nunique():,}")
print(f"Mean rows per video after dedup: {len(deduped_df) / deduped_df['video_id'].nunique():.2f}")

Rows before dedup: 9,876,866
Rows after dedup:  4,183,259
Unique video_id in deduped data: 1,039,987
Mean rows per video after dedup: 4.02


## Encode features (v2 categorical one-hot + `days_to_trend` as a plain numeric)

In [4]:
X_raw_v3 = deduped_df[FEATURE_COLUMNS_RAW_V3]
y = deduped_df[TARGET_COLUMN].astype('float64')
groups = deduped_df['video_id']

X_encoded = pd.get_dummies(X_raw_v3, columns=['video_category_id'], drop_first=True)
FEATURE_COLUMNS = list(X_encoded.columns)
print(f"Feature matrix shape: {X_encoded.shape}")
print(f"Total features: {len(FEATURE_COLUMNS)} (v2 dedup baseline had 19; +1 for days_to_trend)")
print(FEATURE_COLUMNS)

Feature matrix shape: (4183259, 20)
Total features: 20 (v2 dedup baseline had 19; +1 for days_to_trend)
['tag_count', 'title_length', 'title_has_caps_word', 'publish_hour', 'publish_dayofweek', 'days_to_trend', 'video_category_id_Comedy', 'video_category_id_Education', 'video_category_id_Entertainment', 'video_category_id_Film & Animation', 'video_category_id_Gaming', 'video_category_id_Howto & Style', 'video_category_id_Music', 'video_category_id_News & Politics', 'video_category_id_Nonprofits & Activism', 'video_category_id_People & Blogs', 'video_category_id_Pets & Animals', 'video_category_id_Science & Technology', 'video_category_id_Sports', 'video_category_id_Travel & Events']


## GroupShuffleSplit by `video_id` — identical split logic to `04_dedup_group_split.ipynb`

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_encoded, y, groups=groups))

X_train, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
groups_train, groups_val = groups.iloc[train_idx], groups.iloc[val_idx]

print(f"Train: {X_train.shape} ({groups_train.nunique():,} unique videos)")
print(f"Val:   {X_val.shape} ({groups_val.nunique():,} unique videos)")
print(f"Video overlap between train and val: {len(set(groups_train) & set(groups_val)):,} (should be 0)")

Train: (3343308, 20) (831,989 unique videos)
Val:   (839951, 20) (207,998 unique videos)


Video overlap between train and val: 0 (should be 0)


## Train Linear Regression

In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
r2_lr = r2_score(y_val, y_pred_lr)

print('=== Linear Regression (+ days_to_trend) — validation results ===')
print(f'RMSE: {rmse_lr:,.2f} views')
print(f'R^2:  {r2_lr:.4f}')

=== Linear Regression (+ days_to_trend) — validation results ===
RMSE: 6,902,329.34 views
R^2:  0.1114


## Train Random Forest

Same hyperparameters as `04_dedup_group_split.ipynb` (`n_estimators=200, max_depth=15, n_jobs=-1, random_state=42`).

In [7]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
r2_rf = r2_score(y_val, y_pred_rf)

print('=== Random Forest (+ days_to_trend) — validation results ===')
print(f'RMSE: {rmse_rf:,.2f} views')
print(f'R^2:  {r2_rf:.4f}')

=== Random Forest (+ days_to_trend) — validation results ===
RMSE: 6,243,814.23 views
R^2:  0.2729


## Comparison against the dedup-only (no `days_to_trend`) baseline from `04_dedup_group_split.ipynb`

In [8]:
rmse_lr_v2 = 7_152_346.60
r2_lr_v2 = 0.0458
rmse_rf_v2 = 7_011_337.01
r2_rf_v2 = 0.0831

print(f'{"":38s} {"RMSE":>18s} {"R^2":>10s}')
print(f'{"Linear Regression — v2 (no days_to_trend)":38s} {rmse_lr_v2:>18,.2f} {r2_lr_v2:>10.4f}')
print(f'{"Linear Regression — + days_to_trend":38s} {rmse_lr:>18,.2f} {r2_lr:>10.4f}')
print(f'{"Random Forest — v2 (no days_to_trend)":38s} {rmse_rf_v2:>18,.2f} {r2_rf_v2:>10.4f}')
print(f'{"Random Forest — + days_to_trend":38s} {rmse_rf:>18,.2f} {r2_rf:>10.4f}')
print()
print(f'Linear RMSE change: {rmse_lr - rmse_lr_v2:+,.2f} ({(rmse_lr - rmse_lr_v2) / rmse_lr_v2:+.2%})')
print(f'Linear R^2 change:  {r2_lr - r2_lr_v2:+.4f} ({(r2_lr - r2_lr_v2) / r2_lr_v2:+.2%} relative)')
print(f'RF RMSE change:     {rmse_rf - rmse_rf_v2:+,.2f} ({(rmse_rf - rmse_rf_v2) / rmse_rf_v2:+.2%})')
print(f'RF R^2 change:      {r2_rf - r2_rf_v2:+.4f} ({(r2_rf - r2_rf_v2) / r2_rf_v2:+.2%} relative)')

                                                     RMSE        R^2
Linear Regression — v2 (no days_to_trend)       7,152,346.60     0.0458
Linear Regression — + days_to_trend          6,902,329.34     0.1114
Random Forest — v2 (no days_to_trend)        7,011,337.01     0.0831
Random Forest — + days_to_trend              6,243,814.23     0.2729

Linear RMSE change: -250,017.26 (-3.50%)
Linear R^2 change:  +0.0656 (+143.20% relative)
RF RMSE change:     -767,522.78 (-10.95%)
RF R^2 change:      +0.1898 (+228.34% relative)


## `days_to_trend`'s contribution — coefficient (linear) and importance (RF)

In [9]:
coefs = pd.Series(lr.coef_, index=FEATURE_COLUMNS).sort_values(ascending=False)
coef_rank_by_abs = coefs.abs().sort_values(ascending=False).index.get_loc('days_to_trend') + 1

print('=== Linear Regression coefficient for days_to_trend ===')
print(f"days_to_trend coefficient: {coefs['days_to_trend']:,.2f} views per additional day trending")
print(f"Rank by |coefficient| among all {len(FEATURE_COLUMNS)} features: #{coef_rank_by_abs}")
print()
print('Top 5 by |coefficient|:')
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(5))
print()

importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
importance_rank = list(importances.index).index('days_to_trend') + 1

print('=== Random Forest importance for days_to_trend ===')
print(f"days_to_trend importance: {importances['days_to_trend']:.4f} "
      f"({importances['days_to_trend'] / importances.sum():.1%} of total)")
print(f"Rank among all {len(FEATURE_COLUMNS)} features: #{importance_rank}")
print()
print('Top 5 by importance:')
print(importances.head(5))

=== Linear Regression coefficient for days_to_trend ===
days_to_trend coefficient: 262,284.47 views per additional day trending
Rank by |coefficient| among all 20 features: #15

Top 5 by |coefficient|:
video_category_id_Howto & Style           6.079489e+06
video_category_id_Science & Technology    3.599658e+06
video_category_id_Pets & Animals          3.272437e+06
video_category_id_Travel & Events         3.082671e+06
video_category_id_Comedy                  2.670466e+06
dtype: float64

=== Random Forest importance for days_to_trend ===
days_to_trend importance: 0.2131 (21.3% of total)
Rank among all 20 features: #1

Top 5 by importance:
days_to_trend        0.213054
title_length         0.209651
publish_hour         0.158854
tag_count            0.152374
publish_dayofweek    0.082687
dtype: float64


## Save the models

In [10]:
LR_PATH = MODELS_DIR / 'linear_regression_days_to_trend.pkl'
RF_PATH = MODELS_DIR / 'random_forest_days_to_trend.pkl'
FEATURES_PATH = MODELS_DIR / 'days_to_trend_feature_columns.json'

joblib.dump(lr, LR_PATH)
joblib.dump(rf, RF_PATH)
with open(FEATURES_PATH, 'w') as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print(f'Saved Linear Regression to {LR_PATH}')
print(f'Saved Random Forest to {RF_PATH}')
print(f'Saved {len(FEATURE_COLUMNS)} feature columns to {FEATURES_PATH}')

Saved Linear Regression to ../models/linear_regression_days_to_trend.pkl
Saved Random Forest to ../models/random_forest_days_to_trend.pkl
Saved 20 feature columns to ../models/days_to_trend_feature_columns.json


## Summary

**The theory holds.** Adding `days_to_trend` (days between publish and the trending-date snapshot) to the deduped, group-split feature set improved both models meaningfully:

| | RMSE | R² |
|---|---:|---:|
| Linear Regression — v2 (no days_to_trend) | 7,152,347 | 0.0458 |
| Linear Regression — + days_to_trend | 6,902,329 | 0.1114 |
| Random Forest — v2 (no days_to_trend) | 7,011,337 | 0.0831 |
| Random Forest — + days_to_trend | 6,243,814 | 0.2729 |

Linear R² more than doubled in relative terms (+143%, 0.046 → 0.111); Random Forest's more than tripled (+228%, 0.083 → 0.273). RMSE fell too (-3.5% linear, -11.0% RF) — a genuine improvement, not the variance-shrinkage artifact seen when the leaked split was fixed, since the validation set (and its target distribution) is identical here to `04_dedup_group_split.ipynb`; only the feature set changed.

**`days_to_trend` is the single most important feature for Random Forest** — importance 0.213, ranked #1 of 20, ahead of `title_length` (0.210), `publish_hour` (0.159), and `tag_count` (0.152). For Linear Regression, its raw coefficient (+262,284 views per additional day trending) ranks only #15 by magnitude among 20 features — but that's a scale artifact, not a real weakness: `days_to_trend` spans dozens of days on this dataset, so a swing of ~20 days corresponds to ~5.2M predicted views, comparable to the biggest category-dummy coefficients (3–6M). Raw coefficient magnitude isn't directly comparable across features on different scales (same caveat noted in `02_baseline_linear_regression.ipynb`) — the R² jump is the more trustworthy signal of its contribution for the linear model.

**Conclusion**: the hypothesis from `04_dedup_group_split.ipynb` is confirmed — a real, sizeable share of the "unexplained" variance in the deduped/grouped baseline was exactly the missing "which day of the trending run is this" signal, not just noise. `days_to_trend` is not usable as a true pre-publish feature (you can't know it before a video starts trending), but it's valuable for understanding the data-generating process, and for any modeling task that isn't strictly "predict virality before publish" (e.g. "given a video is N days into trending, estimate its current view count").